In [0]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.window import Window

spark = SparkSession.builder.getOrCreate()

emp_data = [
    (1, "Sandeep", "IT", 10000, "2026-07-01", "A1", "sandeep@gmail.com", [101, 102], '{"city":"Bengaluru","skills":["python","sql"]}'),
    (2, "Ravi", "IT", 12000, "2026-07-03", "A2", "ravi@gmail.com", [103], '{"city":"Mumbai","skills":["pyspark","sql"]}'),
    (3, "Anita", "HR", 9000, "2026-07-02", "A3", "anita@gmail.com", [104, 105], '{"city":"Pune","skills":["excel","communication"]}'),
    (4, "Kiran", "HR", 9500, "2026-07-04", "A1", "kiran@gmail.com", [], '{"city":"Delhi","skills":[]}'),
    (5, "Meena", "IT", 12000, "2026-07-05", "A2", "meena@gmail.com", [106], '{"city":"Chennai","skills":["spark","databricks"]}')
]

cols = ["id", "name", "dept", "salary", "join_date", "grade", "email", "product_ids", "profile_json"]
df = spark.createDataFrame(emp_data, cols)
df.show(truncate=False)

In [0]:
w = Window.partitionBy("dept").orderBy(F.col("salary").desc())

df.select(
    "id", "name", "dept", "salary",
    F.row_number().over(w).alias("row_number"),
    F.rank().over(w).alias("rank"),
    F.dense_rank().over(w).alias("dense_rank")
).show()

In [0]:
w2 = Window.partitionBy("dept").orderBy("join_date")

df.select(
    "name", "dept", "join_date",
    F.lag("salary", 1).over(w2).alias("prev_salary"),
    F.lead("salary", 1).over(w2).alias("next_salary")
).show()

In [0]:
w3 = Window.partitionBy("dept").orderBy("join_date").rowsBetween(Window.unboundedPreceding, Window.currentRow)

df.select(
    "name", "dept", "salary", "join_date",
    F.sum("salary").over(w3).alias("running_salary")
).show()

In [0]:
df.select(
    F.trim("name").alias("trimmed_name"),
    F.upper("dept").alias("dept_upper"),
    F.lower("email").alias("email_lower")
).show(truncate=False)

In [0]:
df.select(
    F.concat(F.col("name"), F.lit("_"), F.col("grade")).alias("name_grade"),
    F.concat_ws("-", "dept", "grade").alias("dept_grade")
).show(truncate=False)

In [0]:
df.select(
    F.substring("email", 1, 6).alias("email_prefix"),
    F.split("email", "@").alias("email_parts")
).show(truncate=False)

In [0]:
df.select(
    F.regexp_replace("email", "@gmail.com", "@company.com").alias("new_email")
).show(truncate=False)

In [0]:
num_df = df.select(
    F.round("salary", 0).alias("rounded_salary"),
    F.floor(F.col("salary") / 3).alias("salary_floor"),
    F.ceil(F.col("salary") / 3).alias("salary_ceil"),
    F.abs(F.col("salary") - F.lit(11000)).alias("diff_from_11000")
)
num_df.show()

In [0]:
sdf = spark.createDataFrame([("100.75",), ("200.25",)], ["amount_str"])

sdf.select(
    F.col("amount_str").cast("double").alias("amount_double"),
    F.col("amount_str").cast("int").alias("amount_int")
).show()

In [0]:
spark.createDataFrame([(10, 20, 15), (5, 3, 8)], ["a", "b", "c"]).select(
    F.greatest("a", "b", "c").alias("max_value"),
    F.least("a", "b", "c").alias("min_value")
).show()

In [0]:
df2 = df.withColumn("join_date", F.to_date("join_date"))

df2.printSchema()

In [0]:
df2.select(
    "name", "join_date",
    F.datediff(F.current_date(), "join_date").alias("days_since_join"),
    F.date_add("join_date", 7).alias("after_7_days"),
    F.date_sub("join_date", 3).alias("before_3_days")
).show()

In [0]:
df2.select(
    "name",
    F.date_format("join_date", "dd-MMM-yyyy").alias("formatted_date")
).show()

In [0]:
df.groupBy("dept").agg(
    F.count("*").alias("emp_count"),
    F.sum("salary").alias("total_salary"),
    F.avg("salary").alias("avg_salary"),
    F.min("salary").alias("min_salary"),
    F.max("salary").alias("max_salary")
).show()

In [0]:
df.groupBy("dept").agg(
    F.countDistinct("grade").alias("distinct_grades")
).show()

In [0]:
dept_df = spark.createDataFrame([
    ("IT", "Information Technology"),
    ("HR", "Human Resources")
], ["dept", "dept_full_name"])

In [0]:
df.join(dept_df, on="dept", how="left").show(truncate=False)

In [0]:
df.join(dept_df, on="dept", how="inner").show(truncate=False)

In [0]:
df.join(dept_df.filter(F.col("dept") == "Finance"), on="dept", how="left_anti").show()

In [0]:
df.select(
    "name", "product_ids",
    F.size("product_ids").alias("product_count"),
    F.array_contains("product_ids", 101).alias("has_101")
).show(truncate=False)

In [0]:
df.select(
    "name",
    F.explode("product_ids").alias("product_id")
).show()

In [0]:
df.select(
    "name",
    F.posexplode("product_ids").alias("pos", "product_id")
).show()

In [0]:
from pyspark.sql.types import StructType, StructField, StringType, ArrayType

schema = StructType([
    StructField("city", StringType(), True),
    StructField("skills", ArrayType(StringType()), True)
])

nested_df = df.select("name", F.from_json("profile_json", schema).alias("profile"))
nested_df.show(truncate=False)

In [0]:
nested_df.select(
    "name",
    F.col("profile.city").alias("city"),
    F.col("profile.skills").alias("skills")
).show(truncate=False)

In [0]:
nested_df.select(
    "name",
    F.col("profile.city").alias("city"),
    F.explode("profile.skills").alias("skill")
).show()

In [0]:
nested_df.select("name", "profile.*").show(truncate=False)

In [0]:
schema = StructType([
    StructField("city", StringType(), True),
    StructField("skills", ArrayType(StringType()), True)
])

final_df = (
    df
    .withColumn("join_date", F.to_date("join_date"))
    .withColumn("name_clean", F.trim("name"))
    .withColumn("email_domain", F.split("email", "@").getItem(1))
    .withColumn("profile", F.from_json("profile_json", schema))
    .withColumn("city", F.col("profile.city"))
    .withColumn("skill", F.explode("profile.skills"))
)

w = Window.partitionBy("dept").orderBy(F.col("salary").desc())

final_df.select(
    "name_clean", "dept", "salary", "join_date", "city", "skill", "email_domain",
    F.row_number().over(w).alias("rn"),
    F.rank().over(w).alias("rnk")
).show(truncate=False)